# NepPlural — NLU Baseline: Multi-Task Fine-Tuning

Fine-tunes a Hugging Face encoder on the **Final Annotations** (4 tasks per comment:
`intent`, `primary_driver`, `value_orientation`, `affect`) with a shared encoder + 4 classification heads.

**How to use (Google Colab):**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Edit `MODEL_NAME` in the Config cell — everything else stays the same. Works with any
   `AutoModel`-compatible encoder.
3. Provide the data (Drive mount or zip upload — see Data cell).
4. Run all. Test metrics + predictions are saved under `OUTPUT_DIR`, named after the model.

Notes: mean-pooling over the last hidden state is used (model-agnostic, no reliance on a CLS token).
Loss is class-weighted cross-entropy summed over the 4 tasks to handle label imbalance
(e.g. `Trapped/Regretful` has only ~26 examples). Primary metric: **macro-F1** per task.

In [ ]:
!pip -q install -U transformers scikit-learn pandas

In [ ]:
# ============================== CONFIG ==============================
# Change MODEL_NAME to any Hugging Face encoder — nothing else needs to change.
MODEL_NAME = "xlm-roberta-base"

DATA_DIR   = "/content/Final_Annotations"   # folder containing the *_Cleared.csv files (searched recursively)
OUTPUT_DIR = "/content/outputs"             # metrics + predictions are written here

TASKS      = ["intent", "primary_driver", "value_orientation", "affect"]
TEXT_COL   = "comment"

MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS     = 10
LR         = 1e-5     # XLM-R is unstable on small data at 2e-5; 1e-5 is safer
WARMUP_FRAC = 0.1
DROPOUT    = 0.1
WEIGHT_DAMP = 0.5    # class-weight damping: 1.0 = full inverse-frequency, 0.5 = sqrt, 0.0 = unweighted
SEED       = 42

VAL_FRAC   = 0.10   # of total
TEST_FRAC  = 0.20   # of total
# ====================================================================

In [ ]:
import os, glob, json, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device} | AMP: {use_amp}")

## Data

Two options — use whichever is convenient:

**A. Google Drive** — uncomment the mount lines and point `DATA_DIR` at the folder in your Drive
(e.g. `/content/drive/MyDrive/PluralBench-NP/Final_Annotations`).

**B. Zip upload** — zip the `Final_Annotations` folder locally, run the cell, and upload when prompted.

In [ ]:
# Option A: Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/PluralBench-NP/Final_Annotations"

# Option B: zip upload (runs automatically if DATA_DIR doesn't exist yet)
if not os.path.isdir(DATA_DIR):
    import zipfile
    from google.colab import files
    print("Upload a zip containing the Final_Annotations folder...")
    uploaded = files.upload()
    for name in uploaded:
        with zipfile.ZipFile(name) as z:
            z.extractall("/content")
    assert os.path.isdir(DATA_DIR), f"{DATA_DIR} still not found — check the zip's folder structure."

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
assert csv_files, f"No CSVs found under {DATA_DIR}"
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print("  ", os.path.relpath(f, DATA_DIR))

df = pd.concat(
    [pd.read_csv(f).assign(source=os.path.relpath(f, DATA_DIR)) for f in csv_files],
    ignore_index=True,
)

# Basic cleaning: required columns, drop empty/duplicate comments (avoids train/test leakage)
df = df.dropna(subset=[TEXT_COL] + TASKS)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""]
before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print(f"\nRows: {before} -> {len(df)} after dedup")

# Build label maps dynamically from the data
label2id = {t: {lab: i for i, lab in enumerate(sorted(df[t].unique()))} for t in TASKS}
id2label = {t: {i: lab for lab, i in m.items()} for t, m in label2id.items()}
for t in TASKS:
    df[f"{t}_id"] = df[t].map(label2id[t])
    print(f"\n{t} ({len(label2id[t])} classes):")
    print(df[t].value_counts().to_string())

In [ ]:
# Stratified split on `intent` (the task with the rarest class) -> ~70/10/20
train_df, test_df = train_test_split(
    df, test_size=TEST_FRAC, stratify=df["intent"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=train_df["intent"], random_state=SEED
)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

# Class weights from the train split only (inverse-frequency), per task
class_weights = {}
for t in TASKS:
    counts = train_df[f"{t}_id"].value_counts().sort_index()
    counts = counts.reindex(range(len(label2id[t])), fill_value=1)  # guard: class missing from train
    w = (len(train_df) / (len(counts) * counts.values)) ** WEIGHT_DAMP
    class_weights[t] = torch.tensor(w, dtype=torch.float32, device=device)
    print(f"{t} weights: {np.round(w, 2)}")
# Sanity baseline: majority-class macro-F1 on the test split — the model must clearly beat this
print("\nMajority-class baseline (test macro-F1):")
baseline = {}
for t in TASKS:
    maj = train_df[f"{t}_id"].mode()[0]
    baseline[t] = f1_score(test_df[f"{t}_id"], [maj] * len(test_df), average="macro")
    print(f"  {t}: {baseline[t]:.3f}")
print(f"  mean: {np.mean(list(baseline.values())):.3f}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame[TEXT_COL].tolist()
        self.labels = {t: frame[f"{t}_id"].tolist() for t in TASKS}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        item = {"text": self.texts[i]}
        for t in TASKS:
            item[t] = self.labels[t][i]
        return item

def collate(batch):
    enc = tokenizer(
        [b["text"] for b in batch],
        truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt",
    )
    labels = {t: torch.tensor([b[t] for b in batch], dtype=torch.long) for t in TASKS}
    return enc, labels

train_loader = DataLoader(CommentDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(CommentDataset(val_df),   batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(CommentDataset(test_df),  batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate)

In [ ]:
class MultiTaskClassifier(nn.Module):
    """Shared encoder + one linear head per task. Mean-pooled, model-agnostic."""

    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, n) for t, n in num_labels.items()})

    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = self.dropout(pooled)
        return {t: head(pooled) for t, head in self.heads.items()}

model = MultiTaskClassifier(
    MODEL_NAME, {t: len(label2id[t]) for t in TASKS}, dropout=DROPOUT
).to(device)
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds = {t: [] for t in TASKS}
    golds = {t: [] for t in TASKS}
    for enc, labels in loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(**enc)
        for t in TASKS:
            preds[t].extend(logits[t].argmax(-1).cpu().tolist())
            golds[t].extend(labels[t].tolist())
    macro_f1 = {t: f1_score(golds[t], preds[t], average="macro") for t in TASKS}
    return macro_f1, preds, golds

total_steps = len(train_loader) * EPOCHS
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(WARMUP_FRAC * total_steps), total_steps
)
scaler = torch.amp.GradScaler(enabled=use_amp)
loss_fns = {t: nn.CrossEntropyLoss(weight=class_weights[t]) for t in TASKS}

best_score, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, (enc, labels) in enumerate(train_loader, 1):
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = {t: v.to(device) for t, v in labels.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(**enc)
            loss = sum(loss_fns[t](logits[t], labels[t]) for t in TASKS)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()

    val_f1, _, _ = evaluate(val_loader)
    mean_f1 = float(np.mean(list(val_f1.values())))
    flag = ""
    if mean_f1 > best_score:
        best_score = mean_f1
        best_state = copy.deepcopy(model.state_dict())
        flag = "  <- best"
    detail = "  ".join(f"{t}={val_f1[t]:.3f}" for t in TASKS)
    print(f"epoch {epoch}/{EPOCHS}  loss={running/len(train_loader):.4f}  "
          f"val macro-F1: mean={mean_f1:.3f}  [{detail}]{flag}")

model.load_state_dict(best_state)
print(f"\nRestored best checkpoint (val mean macro-F1 = {best_score:.3f})")

In [ ]:
# ----- Test evaluation -----
test_f1, test_preds, test_golds = evaluate(test_loader)

print(f"MODEL: {MODEL_NAME}\n" + "=" * 60)
summary = {}
for t in TASKS:
    acc = accuracy_score(test_golds[t], test_preds[t])
    summary[t] = {"macro_f1": round(test_f1[t], 4), "accuracy": round(acc, 4)}
    print(f"\n### {t}  |  macro-F1 = {test_f1[t]:.4f}  acc = {acc:.4f}")
    names = [id2label[t][i] for i in range(len(id2label[t]))]
    print(classification_report(
        test_golds[t], test_preds[t],
        labels=list(range(len(names))), target_names=names,
        digits=3, zero_division=0,
    ))

summary["mean_macro_f1"] = round(float(np.mean(list(test_f1.values()))), 4)
print("=" * 60)
print(f"MEAN macro-F1 over 4 tasks: {summary['mean_macro_f1']:.4f}")

In [ ]:
# ----- Save metrics, predictions, and label maps -----
os.makedirs(OUTPUT_DIR, exist_ok=True)
tag = MODEL_NAME.replace("/", "__")

with open(f"{OUTPUT_DIR}/{tag}_metrics.json", "w") as f:
    json.dump({
        "model": MODEL_NAME, "seed": SEED, "epochs": EPOCHS, "lr": LR,
        "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
        "best_val_mean_macro_f1": round(best_score, 4),
        "test": summary,
    }, f, indent=2)

pred_df = test_df[[TEXT_COL, "source"] + TASKS].copy()
for t in TASKS:
    pred_df[f"{t}_pred"] = [id2label[t][p] for p in test_preds[t]]
    pred_df[f"{t}_correct"] = pred_df[t] == pred_df[f"{t}_pred"]
pred_df.to_csv(f"{OUTPUT_DIR}/{tag}_test_predictions.csv", index=False)

with open(f"{OUTPUT_DIR}/label_maps.json", "w") as f:
    json.dump(label2id, f, indent=2, ensure_ascii=False)

torch.save(best_state, f"{OUTPUT_DIR}/{tag}_best.pt")
print(f"Saved to {OUTPUT_DIR}/:")
print(f"  {tag}_metrics.json")
print(f"  {tag}_test_predictions.csv")
print(f"  {tag}_best.pt")
print("  label_maps.json")

## Comparing models

- To benchmark another model, change `MODEL_NAME` in the Config cell and re-run all cells —
  the split is deterministic (`SEED`), so every model sees the exact same train/val/test data.
- Each run leaves a `*_metrics.json` in `OUTPUT_DIR`, so you can collect them into a results table:

```python
import glob, json, pandas as pd
rows = [json.load(open(f)) for f in glob.glob(f"{OUTPUT_DIR}/*_metrics.json")]
pd.json_normalize(rows)[["model", "seed", "test.mean_macro_f1",
    "test.intent.macro_f1", "test.primary_driver.macro_f1",
    "test.value_orientation.macro_f1", "test.affect.macro_f1"]]
```

- The same test split should be used for the zero-shot LLM evaluation (`prompts/eval_prompt.md`)
  so the fine-tuned and zero-shot numbers are directly comparable — export it with:
  `test_df[[TEXT_COL] + TASKS].to_csv(f"{OUTPUT_DIR}/test_split.csv", index=False)`